# Gold functions (definitions only)
Silver to Gold transformations for the star schema. This notebook only **defines** functions,
so it is safe to `%run` from `driver_session` (Step 4) or from `gold_load`.

Run `./common` first. The entry point is `run_gold(batch_id, run_date, load_type)`.

**How incremental selection works:** each Silver row is "new" when its `load_timestamp`
(`snapshot_at` for repo metadata) is later than the GOLD watermark for that repository and table,
and not later than the maximum value seen when the run started. Watermarks only advance after
every step succeeds, and every MERGE is idempotent, so a failed run can simply be re-run.

In [ ]:
import uuid
import time
import traceback
from datetime import datetime

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window

if 'log' not in globals() or 'WM_TABLE' not in globals():
    raise RuntimeError('Run ./common before ./gold_functions')

spark.conf.set('spark.sql.session.timeZone', 'UTC')

GOLD = f'{CATALOG}.gold'
SILVER = f'{CATALOG}.silver'

# Silver table -> column that marks a row as new for Gold
GOLD_SOURCES = {
    'commits': 'load_timestamp',
    'issues': 'load_timestamp',
    'pulls': 'load_timestamp',
    'repo_metadata': 'snapshot_at',
    'contributors': 'load_timestamp',
}

GOLD_TABLES = [
    'dim_date', 'dim_repository', 'dim_contributor',
    'fact_commits', 'fact_issues', 'fact_pull_requests',
    'fact_repo_daily_activity', 'fact_contributor_activity', 'fact_repo_snapshot',
]

GOLD_EPOCH = datetime(1970, 1, 1)
UNKNOWN_KEY = -1
print('Gold functions loading for', GOLD)

## Helpers

In [ ]:
def date_key(column):
    """yyyymmdd integer key from a timestamp column (NULL stays NULL)."""
    return F.date_format(F.col(column), 'yyyyMMdd').cast('int')


def latest_version(table):
    return spark.sql(f'DESCRIBE HISTORY {table} LIMIT 1').first()['version']


def merge_metrics(table, before_version):
    """Inserted/updated counts from the MERGE commits made after `before_version`.

    Using the version avoids reading an older MERGE when an empty MERGE writes no commit.
    """
    rows = (spark.sql(f'DESCRIBE HISTORY {table} LIMIT 20')
            .filter(f"version > {before_version} AND operation = 'MERGE'")
            .collect())
    inserted = sum(int(r['operationMetrics'].get('numTargetRowsInserted', 0) or 0) for r in rows)
    updated = sum(int(r['operationMetrics'].get('numTargetRowsUpdated', 0) or 0) for r in rows)
    return inserted, updated


def merge_gold(source, table, keys, compare, update_condition=None):
    """MERGE `source` into `table`.

    Insert new keys. Update existing rows only when a compared business column changed
    (or when `update_condition` is true), so identical re-runs rewrite nothing.
    """
    columns = source.columns
    on = ' AND '.join(f't.{k} = s.{k}' for k in keys)
    changed = update_condition or ' OR '.join(f'NOT (t.{c} <=> s.{c})' for c in compare)
    before = latest_version(table)
    (DeltaTable.forName(spark, table).alias('t')
        .merge(source.alias('s'), on)
        .whenMatchedUpdate(condition=changed,
                           set={c: f's.`{c}`' for c in columns if c not in keys})
        .whenNotMatchedInsert(values={c: f's.`{c}`' for c in columns})
        .execute())
    return merge_metrics(table, before)


def check_keys(df, column, dim_table, dim_key, label):
    """Fail loudly if a foreign key is NULL or has no row in its dimension."""
    dim = spark.table(dim_table).select(F.col(dim_key).alias('_dim_key'))
    nulls = df.filter(F.col(column).isNull()).count()
    orphans = (df.filter(F.col(column).isNotNull())
                 .join(F.broadcast(dim), F.col(column) == F.col('_dim_key'), 'left_anti'))
    sample = [r[0] for r in orphans.select(column).distinct().limit(5).collect()]
    if nulls or sample:
        raise ValueError(
            f'{label}: {nulls} NULL keys; orphan keys such as {sample} '
            f'have no match in {dim_table}')


def with_repository_key(df):
    """Attach repository_key by joining Silver's repository name to dim_repository."""
    dim = (spark.table(f'{GOLD}.dim_repository')
           .select('repository_key', F.col('repository_name').alias('_repo_name')))
    return (df.join(F.broadcast(dim), F.col('repository') == F.col('_repo_name'), 'left')
              .drop('_repo_name'))

## Watermarks and selecting new Silver rows

In [ ]:
def read_gold_watermarks():
    rows = (spark.table(WM_TABLE).filter("layer = 'GOLD'")
            .select('repository', 'entity', 'last_successful_watermark').collect())
    return {(r['entity'], r['repository']): r['last_successful_watermark'] for r in rows}


def read_uppers():
    """Highest marker value per Silver table and repository, taken once at run start."""
    uppers = {}
    for entity, column in GOLD_SOURCES.items():
        rows = (spark.table(f'{SILVER}.{entity}')
                .groupBy('repository').agg(F.max(column).alias('upper')).collect())
        uppers[entity] = {r['repository']: r['upper'] for r in rows}
    return uppers


def select_new(entity, watermarks, uppers):
    """Silver rows with watermark < marker <= upper, per repository."""
    column = GOLD_SOURCES[entity]
    table = spark.table(f'{SILVER}.{entity}')
    bounds = []
    for repo, upper in uppers[entity].items():
        watermark = watermarks.get((entity, repo), GOLD_EPOCH)
        if upper is not None and upper > watermark:
            bounds.append((repo, watermark, upper))
    if not bounds:
        return table.limit(0)
    bounds_df = spark.createDataFrame(bounds, 'repository string, wm timestamp, upper timestamp')
    return (table.join(F.broadcast(bounds_df), 'repository')
                 .filter((F.col(column) > F.col('wm')) & (F.col(column) <= F.col('upper')))
                 .drop('wm', 'upper'))


def advance_watermarks(uppers, batch_id):
    """Advance all GOLD watermarks in ONE atomic MERGE. Never rewinds."""
    rows = [(repo, entity, 'GOLD', upper, batch_id, now())
            for entity, per_repo in uppers.items()
            for repo, upper in per_repo.items() if upper is not None]
    if not rows:
        return 0
    source = spark.createDataFrame(rows, WM_SCHEMA)
    (DeltaTable.forName(spark, WM_TABLE).alias('t')
        .merge(source.alias('s'),
               't.repository = s.repository AND t.entity = s.entity AND t.layer = s.layer')
        .whenMatchedUpdate(
            condition='t.last_successful_watermark < s.last_successful_watermark',
            set={'last_successful_watermark': 's.last_successful_watermark',
                 'batch_id': 's.batch_id', 'updated_at': 's.updated_at'})
        .whenNotMatchedInsertAll()
        .execute())
    return len(rows)

## Dimension loaders

In [ ]:
def load_dim_repository(new):
    split = F.split(F.col('repository'), '/')
    src = (new['repo_metadata'].select(
        F.col('repo_id').alias('repository_key'),
        F.col('repository').alias('repository_name'),
        split.getItem(0).alias('owner'),
        split.getItem(1).alias('repo_name'),
        'description', 'language', 'default_branch', 'html_url',
        'created_at', 'archived',
        F.current_timestamp().alias('load_timestamp'),
    ).dropDuplicates(['repository_key']))
    rows = src.count()
    if rows == 0:
        return {'rows': 0, 'inserted': 0, 'updated': 0}
    ins, upd = merge_gold(
        src, f'{GOLD}.dim_repository', ['repository_key'],
        ['repository_name', 'owner', 'repo_name', 'description', 'language',
         'default_branch', 'html_url', 'created_at', 'archived'])
    return {'rows': rows, 'inserted': ins, 'updated': upd}


def load_dim_contributor(new):
    # Every author seen in the new activity rows, with their earliest activity date
    activity = (
        new['commits'].select(F.col('author_user_id').alias('contributor_key'),
                              F.to_date('commit_timestamp').alias('d'))
        .unionByName(new['issues'].select(F.col('author_user_id').alias('contributor_key'),
                                          F.to_date('created_at').alias('d')))
        .unionByName(new['pulls'].select(F.col('author_user_id').alias('contributor_key'),
                                         F.to_date('created_at').alias('d')))
        .filter(F.col('contributor_key').isNotNull())
        .groupBy('contributor_key').agg(F.min('d').alias('first_activity_date')))
    # Account type is only known for contributors in the contributors snapshot
    accounts = (new['contributors']
                .select(F.col('contributor_id').alias('contributor_key'), 'account_type')
                .groupBy('contributor_key').agg(F.max('account_type').alias('account_type')))
    src = (activity.join(accounts, 'contributor_key', 'full_outer')
           .filter(F.col('contributor_key') != UNKNOWN_KEY)
           .withColumn('load_timestamp', F.current_timestamp()))
    rows = src.count()
    if rows == 0:
        return {'rows': 0, 'inserted': 0, 'updated': 0}

    table = f'{GOLD}.dim_contributor'
    before = latest_version(table)
    (DeltaTable.forName(spark, table).alias('t')
        .merge(src.alias('s'), 't.contributor_key = s.contributor_key')
        .whenMatchedUpdate(
            condition=('NOT (t.account_type <=> coalesce(s.account_type, t.account_type)) '
                       'OR NOT (t.first_activity_date <=> '
                       'least(t.first_activity_date, s.first_activity_date))'),
            set={'account_type': 'coalesce(s.account_type, t.account_type)',
                 'is_bot': "coalesce(s.account_type, t.account_type) = 'Bot'",
                 'first_activity_date': 'least(t.first_activity_date, s.first_activity_date)',
                 'load_timestamp': 's.load_timestamp'})
        .whenNotMatchedInsert(values={
            'contributor_key': 's.contributor_key',
            'account_type': "coalesce(s.account_type, 'Unknown')",
            'is_bot': "coalesce(s.account_type, 'Unknown') = 'Bot'",
            'first_activity_date': 's.first_activity_date',
            'load_timestamp': 's.load_timestamp'})
        .execute())
    ins, upd = merge_metrics(table, before)
    return {'rows': rows, 'inserted': ins, 'updated': upd}

## Fact loaders (MERGE)

In [ ]:
def load_fact(table, src, compare, key_checks):
    rows = src.count()
    if rows == 0:
        return {'rows': 0, 'inserted': 0, 'updated': 0}
    for column, dim, dim_key in key_checks:
        check_keys(src, column, f'{GOLD}.{dim}', dim_key, f'{table}.{column}')
    ins, upd = merge_gold(src, f'{GOLD}.{table}', ['record_key'], compare)
    return {'rows': rows, 'inserted': ins, 'updated': upd}


def load_fact_commits(new):
    src = with_repository_key(new['commits']).select(
        'record_key', 'commit_sha', 'repository_key',
        F.coalesce(F.col('author_user_id'), F.lit(UNKNOWN_KEY)).cast('bigint').alias('contributor_key'),
        date_key('commit_timestamp').alias('commit_date_key'),
        'commit_timestamp',
        F.coalesce(F.length('message'), F.lit(0)).alias('message_length'),
        F.lit(1).alias('commit_count'),
        'batch_id',
        F.current_timestamp().alias('load_timestamp'))
    return load_fact(
        'fact_commits', src,
        ['commit_sha', 'repository_key', 'contributor_key', 'commit_date_key',
         'commit_timestamp', 'message_length'],
        [('repository_key', 'dim_repository', 'repository_key'),
         ('commit_date_key', 'dim_date', 'date_key')])


def load_fact_issues(new):
    hours = (F.unix_timestamp('closed_at') - F.unix_timestamp('created_at')) / 3600.0
    src = with_repository_key(new['issues']).select(
        'record_key',
        F.col('issue_number'),
        'repository_key',
        F.coalesce(F.col('author_user_id'), F.lit(UNKNOWN_KEY)).cast('bigint').alias('contributor_key'),
        date_key('created_at').alias('created_date_key'),
        F.coalesce(date_key('closed_at'), F.lit(UNKNOWN_KEY)).alias('closed_date_key'),
        'state',
        F.col('closed_at').isNotNull().alias('is_closed'),
        hours.cast('double').alias('hours_to_close'),
        'created_at', 'closed_at', 'updated_at',
        'batch_id',
        F.current_timestamp().alias('load_timestamp'))
    return load_fact(
        'fact_issues', src,
        ['issue_number', 'repository_key', 'contributor_key', 'created_date_key',
         'closed_date_key', 'state', 'is_closed', 'hours_to_close',
         'created_at', 'closed_at', 'updated_at'],
        [('repository_key', 'dim_repository', 'repository_key'),
         ('created_date_key', 'dim_date', 'date_key'),
         ('closed_date_key', 'dim_date', 'date_key')])


def load_fact_pull_requests(new):
    to_merge = (F.unix_timestamp('merged_at') - F.unix_timestamp('created_at')) / 3600.0
    to_close = (F.unix_timestamp('closed_at') - F.unix_timestamp('created_at')) / 3600.0
    src = with_repository_key(new['pulls']).select(
        'record_key',
        F.col('pr_number'),
        'repository_key',
        F.coalesce(F.col('author_user_id'), F.lit(UNKNOWN_KEY)).cast('bigint').alias('contributor_key'),
        date_key('created_at').alias('created_date_key'),
        F.coalesce(date_key('closed_at'), F.lit(UNKNOWN_KEY)).alias('closed_date_key'),
        F.coalesce(date_key('merged_at'), F.lit(UNKNOWN_KEY)).alias('merged_date_key'),
        'state',
        F.coalesce(F.col('draft'), F.lit(False)).alias('is_draft'),
        F.col('merged_at').isNotNull().alias('is_merged'),
        to_merge.cast('double').alias('hours_to_merge'),
        to_close.cast('double').alias('hours_to_close'),
        'created_at', 'closed_at', 'merged_at',
        'batch_id',
        F.current_timestamp().alias('load_timestamp'))
    return load_fact(
        'fact_pull_requests', src,
        ['pr_number', 'repository_key', 'contributor_key', 'created_date_key',
         'closed_date_key', 'merged_date_key', 'state', 'is_draft', 'is_merged',
         'hours_to_merge', 'hours_to_close', 'created_at', 'closed_at', 'merged_at'],
        [('repository_key', 'dim_repository', 'repository_key'),
         ('created_date_key', 'dim_date', 'date_key'),
         ('closed_date_key', 'dim_date', 'date_key'),
         ('merged_date_key', 'dim_date', 'date_key')])


def load_fact_repo_snapshot(new):
    ranking = Window.partitionBy('repository_key', 'snapshot_date_key') \
                    .orderBy(F.col('snapshot_at').desc())
    src = (with_repository_key(new['repo_metadata']).select(
        'repository_key',
        date_key('snapshot_at').alias('snapshot_date_key'),
        'stars_count', 'forks_count', 'open_issues_count', 'subscribers_count',
        'snapshot_at',
        F.current_timestamp().alias('load_timestamp'))
        .withColumn('_rn', F.row_number().over(ranking))
        .filter('_rn = 1').drop('_rn'))
    rows = src.count()
    if rows == 0:
        return {'rows': 0, 'inserted': 0, 'updated': 0}
    check_keys(src, 'repository_key', f'{GOLD}.dim_repository', 'repository_key',
               'fact_repo_snapshot.repository_key')
    check_keys(src, 'snapshot_date_key', f'{GOLD}.dim_date', 'date_key',
               'fact_repo_snapshot.snapshot_date_key')
    # Latest snapshot of the day wins
    ins, upd = merge_gold(
        src, f'{GOLD}.fact_repo_snapshot', ['repository_key', 'snapshot_date_key'], [],
        update_condition='s.snapshot_at > t.snapshot_at')
    return {'rows': rows, 'inserted': ins, 'updated': upd}

## Aggregate facts (recomputed from the Gold facts, then overwritten)
`load_daily_activity` is the most complex transformation: three fact scans, a six-way union,
a partial aggregate, a shuffle on (repository_key, date_key) and a final aggregate.
It is the best candidate for the Spark UI DAG screenshot in the report.

In [ ]:
MEASURES = ['commits', 'issues_opened', 'issues_closed',
            'prs_opened', 'prs_merged', 'prs_closed_unmerged']


def _event(df, date_column, measure, condition=None):
    """One row per source record, with a 1 in the measure it contributes to."""
    if condition is not None:
        df = df.filter(condition)
    return df.select(F.col('repository_key'), F.col(date_column).alias('date_key'),
                     *[F.lit(1 if m == measure else 0).alias(m) for m in MEASURES])


def load_daily_activity():
    fc = spark.table(f'{GOLD}.fact_commits')
    fi = spark.table(f'{GOLD}.fact_issues')
    fp = spark.table(f'{GOLD}.fact_pull_requests')
    events = (
        _event(fc, 'commit_date_key', 'commits')
        .unionByName(_event(fi, 'created_date_key', 'issues_opened'))
        .unionByName(_event(fi, 'closed_date_key', 'issues_closed',
                            F.col('closed_date_key') != UNKNOWN_KEY))
        .unionByName(_event(fp, 'created_date_key', 'prs_opened'))
        .unionByName(_event(fp, 'merged_date_key', 'prs_merged',
                            F.col('merged_date_key') != UNKNOWN_KEY))
        .unionByName(_event(fp, 'closed_date_key', 'prs_closed_unmerged',
                            (F.col('closed_date_key') != UNKNOWN_KEY) & (~F.col('is_merged')))))
    agg = (events.groupBy('repository_key', 'date_key')
           .agg(*[F.sum(m).cast('int').alias(m) for m in MEASURES],
                F.count(F.lit(1)).cast('int').alias('rows_aggregated'))
           .withColumn('net_issue_change', F.col('issues_opened') - F.col('issues_closed'))
           .withColumn('load_timestamp', F.current_timestamp()))
    target = f'{GOLD}.fact_repo_daily_activity'
    agg.select(*spark.table(target).columns).write.insertInto(target, overwrite=True)
    stats = (spark.table(target)
             .agg(F.count(F.lit(1)).alias('n'), F.sum('rows_aggregated').alias('src')).first())
    return {'rows': int(stats['src'] or 0), 'inserted': int(stats['n']), 'updated': 0}


def load_contributor_activity():
    fc = spark.table(f'{GOLD}.fact_commits').filter(F.col('contributor_key') != UNKNOWN_KEY)
    fi = spark.table(f'{GOLD}.fact_issues').filter(F.col('contributor_key') != UNKNOWN_KEY)
    fp = spark.table(f'{GOLD}.fact_pull_requests').filter(F.col('contributor_key') != UNKNOWN_KEY)

    def part(df, date_column, commits, issues, prs):
        return df.select('repository_key', 'contributor_key',
                         F.col(date_column).alias('date_key'),
                         F.lit(commits).alias('commits'),
                         F.lit(issues).alias('issues_opened'),
                         F.lit(prs).alias('prs_opened'))

    events = (part(fc, 'commit_date_key', 1, 0, 0)
              .unionByName(part(fi, 'created_date_key', 0, 1, 0))
              .unionByName(part(fp, 'created_date_key', 0, 0, 1)))
    agg = (events.groupBy('repository_key', 'contributor_key', 'date_key')
           .agg(F.sum('commits').cast('int').alias('commits'),
                F.sum('issues_opened').cast('int').alias('issues_opened'),
                F.sum('prs_opened').cast('int').alias('prs_opened'))
           .withColumn('total_activities',
                       F.col('commits') + F.col('issues_opened') + F.col('prs_opened'))
           .withColumn('load_timestamp', F.current_timestamp()))
    target = f'{GOLD}.fact_contributor_activity'
    agg.select(*spark.table(target).columns).write.insertInto(target, overwrite=True)
    stats = (spark.table(target)
             .agg(F.count(F.lit(1)).alias('n'), F.sum('total_activities').alias('src')).first())
    return {'rows': int(stats['src'] or 0), 'inserted': int(stats['n']), 'updated': 0}

## Step runner with audit logging
Every step writes one row to `ops.pipeline_execution_logs` with `layer = 'GOLD'`:
start and end time, source rows read (rows aggregated for the aggregate facts),
inserted and updated rows, and the traceback on failure.

In [ ]:
def run_step(ctx, table, fn):
    started = now()
    execution_id = str(uuid.uuid4())
    t0 = time.perf_counter()
    try:
        out = fn()
    except Exception:
        error = traceback.format_exc()
        print(f'GOLD {table} FAILED:\n{error}', flush=True)
        try:
            log(spark, execution_id, ctx['batch_id'], 'silver_to_gold', 'GOLD',
                ctx['run_date'], ctx['load_type'], started, 'FAILED', 0, error,
                entity_type=table)
        except Exception as exc:
            print('GOLD audit failure:', repr(exc), flush=True)
        raise
    elapsed = time.perf_counter() - t0
    log(spark, execution_id, ctx['batch_id'], 'silver_to_gold', 'GOLD',
        ctx['run_date'], ctx['load_type'], started, 'SUCCESS', out['rows'],
        upserted=out['inserted'] + out['updated'], entity_type=table,
        inserted=out['inserted'], updated=out['updated'])
    print(f"GOLD {table:28} source_rows={out['rows']:<8,} inserted={out['inserted']:<7,} "
          f"updated={out['updated']:<7,} ({elapsed:.1f}s)", flush=True)
    ctx['results'].append({'table': table, **out, 'seconds': round(elapsed, 1)})
    return out

## Entry point

In [ ]:
def run_gold(batch_id, run_date, load_type='INCREMENTAL', force_aggregates=False):
    """Refresh the whole Gold layer from Silver. Returns a summary dict."""
    for table in GOLD_TABLES:
        if not spark.catalog.tableExists(f'{GOLD}.{table}'):
            raise RuntimeError(f'Missing {GOLD}.{table}. Run gold_setup first.')

    ctx = {'batch_id': batch_id, 'run_date': run_date, 'load_type': load_type, 'results': []}
    pipeline_start = now()
    pipeline_id = str(uuid.uuid4())
    print(f'\n=========== GOLD: batch {batch_id} ({load_type}) ===========', flush=True)

    try:
        watermarks = read_gold_watermarks()
        uppers = read_uppers()
        new = {entity: select_new(entity, watermarks, uppers) for entity in GOLD_SOURCES}

        # 1. Dimensions first: facts need their keys
        run_step(ctx, 'dim_repository', lambda: load_dim_repository(new))
        run_step(ctx, 'dim_contributor', lambda: load_dim_contributor(new))

        # 2. Transaction / accumulating facts
        fact_outs = [
            run_step(ctx, 'fact_commits', lambda: load_fact_commits(new)),
            run_step(ctx, 'fact_issues', lambda: load_fact_issues(new)),
            run_step(ctx, 'fact_pull_requests', lambda: load_fact_pull_requests(new)),
        ]
        facts_changed = sum(o['inserted'] + o['updated'] for o in fact_outs) > 0

        # 3. Aggregates: recompute only when a fact changed (or on first run / forced)
        def maybe(table, loader):
            empty = spark.table(f'{GOLD}.{table}').limit(1).count() == 0
            if force_aggregates or facts_changed or empty:
                return loader()
            print(f'GOLD {table}: no fact changes, aggregate left as is', flush=True)
            return {'rows': 0, 'inserted': 0, 'updated': 0}

        run_step(ctx, 'fact_repo_daily_activity',
                 lambda: maybe('fact_repo_daily_activity', load_daily_activity))
        run_step(ctx, 'fact_contributor_activity',
                 lambda: maybe('fact_contributor_activity', load_contributor_activity))

        # 4. Periodic snapshot
        run_step(ctx, 'fact_repo_snapshot', lambda: load_fact_repo_snapshot(new))

        # 5. Only now advance the watermarks (one atomic MERGE)
        advance_watermarks(uppers, batch_id)

        results = ctx['results']
        log(spark, pipeline_id, batch_id, 'silver_to_gold', 'GOLD', run_date, load_type,
            pipeline_start, 'SUCCESS', sum(r['rows'] for r in results),
            upserted=sum(r['inserted'] + r['updated'] for r in results),
            entity_type='ALL')
    except Exception:
        error = traceback.format_exc()
        try:
            log(spark, pipeline_id, batch_id, 'silver_to_gold', 'GOLD', run_date, load_type,
                pipeline_start, 'FAILED', sum(r['rows'] for r in ctx['results']), error,
                entity_type='ALL')
        except Exception as exc:
            print('GOLD pipeline audit failure:', repr(exc), flush=True)
        raise

    summary = {
        'status': 'SUCCESS',
        'batch_id': batch_id,
        'load_type': load_type,
        'gold_source_rows': sum(r['rows'] for r in results),
        'gold_inserted': sum(r['inserted'] for r in results),
        'gold_updated': sum(r['updated'] for r in results),
        'gold_seconds': round(sum(r['seconds'] for r in results), 1),
    }
    print('\n=========== GOLD COMPLETE ===========', flush=True)
    print(summary, flush=True)
    return summary

print('Gold functions ready: run_gold(batch_id, run_date, load_type)')